# Camb.ai TTS API Testing

This notebook tests the direct HTTP integration for Camb.ai TTS API based on the documentation provided.

In [3]:
import os
import asyncio
import aiohttp
import io
import wave
import numpy as np
import sounddevice as sd

# Set your API key here or make sure it is in your environmen

In [12]:
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
api_key = os.environ.get("CAMB_API_KEY")

In [14]:
print(api_key)

f81439cc-1c39-4c32-ace3-a58bc258e8eb


In [15]:
async def list_voices():
    api_key = os.environ.get("CAMB_API_KEY")
    url = "https://client.camb.ai/apis/list-voices"
    headers = {"x-api-key": api_key}
    
    async with aiohttp.ClientSession() as session:
        async with session.get(url, headers=headers) as resp:
            if resp.status == 200:
                return await resp.json()
            else:
                raise Exception(f"Error: {resp.status}\n{await resp.text()}")

# Test listing voices
voices = await list_voices()
print(f"Found {len(voices)} voices:\n")
for voice in voices[:10]:
    print(f"ID: {voice['id']}, Name: {voice['voice_name']}, Gender: {voice.get('gender')}")

Found 796 voices:

ID: 163732, Name: anime_audio_1, Gender: 2
ID: 170512, Name: Lennart Kurz, Gender: 1
ID: 170539, Name: Greta Ackermann, Gender: 2
ID: 170542, Name: Lara Falk, Gender: 2
ID: 170544, Name: Dominik Hauser, Gender: 1
ID: 147319, Name: Silas Blackwood, Gender: 1
ID: 147320, Name: Winston Farnsworth, Gender: 1
ID: 147321, Name: Victor Thorne, Gender: 1
ID: 147324, Name: Toby Parker, Gender: 1
ID: 147325, Name: Marcus Steele, Gender: 1


In [16]:
async def stream_tts(text: str, voice_id: int = 147320):
    api_key = os.environ.get("CAMB_API_KEY")
    url = "https://client.camb.ai/apis/tts-stream"
    
    headers = {
        "x-api-key": api_key,
        "Content-Type": "application/json",
    }
    
    payload = {
        "text": text,
        "voice_id": voice_id,
        "language": "en-us",
        "speech_model": "mars-8.1-flash-beta",
        "output_configuration": {"format": "wav"},
    }
    
    timeout = aiohttp.ClientTimeout(total=120)
    
    async with aiohttp.ClientSession(timeout=timeout) as session:
        async with session.post(url, headers=headers, json=payload) as resp:
            print(f"Status: {resp.status}")
            print(f"Content-Type: {resp.headers.get('Content-Type')}")
            print(f"X-Credits-Required: {resp.headers.get('X-Credits-Required')}")
            resp.raise_for_status()
            
            async for chunk in resp.content.iter_chunked(4096):
                yield chunk

# Stream to a file
with open("camb_streamed_output.wav", "wb") as f:
    async for chunk in stream_tts("[excited] Hello! This is streamed audio generation directly from the Camb AI API."):
        f.write(chunk)
print("Audio saved to camb_streamed_output.wav")

Status: 200
Content-Type: audio/wav
X-Credits-Required: 8.666999999999999815258888702
Audio saved to camb_streamed_output.wav


In [17]:
import IPython.display as ipd

# Play the saved file in Jupyter
ipd.Audio("camb_streamed_output.wav")

In [18]:
async def play_tts(text: str):
    api_key = os.environ.get("CAMB_API_KEY")
    url = "https://client.camb.ai/apis/tts-stream"
    
    headers = {
        "x-api-key": api_key,
        "Content-Type": "application/json",
    }
    
    payload = {
        "text": text,
        "voice_id": 147320,
        "language": "en-us",
        "speech_model": "mars-8.1-flash-beta",
        "output_configuration": {"format": "wav"},
    }
    
    timeout = aiohttp.ClientTimeout(total=120)
    
    async with aiohttp.ClientSession(timeout=timeout) as session:
        async with session.post(url, headers=headers, json=payload) as resp:
            resp.raise_for_status()
            audio_bytes = await resp.read()

    with wave.open(io.BytesIO(audio_bytes), 'rb') as wav_file:
        sample_rate = wav_file.getframerate()
        audio_data = np.frombuffer(
            wav_file.readframes(-1),
            dtype=np.int16
        )

    print("Playing audio with sounddevice...")
    sd.play(audio_data, samplerate=sample_rate)
    sd.wait()
    print("Done.")

await play_tts("[speaking slowly] This is very important. Let's pause <break time=\"400ms\"/> and continue.")

Playing audio with sounddevice...
Done.
